In [1]:
# imports

import os, re, json, pickle, torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress minor warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

### data crawl
* preprocessing included in moviedataorchestrator 
    * clean currency
    * normalize rating 

In [6]:
import os
import re
import pandas as pd
import numpy as np

class MovieWorkflowManager:
    def __init__(self):
        self.base_path = os.getcwd() 
        self.folders = {
            "audience-reviews":      {"type": "audience", "platform": "rt"},
            "critic-reviews":        {"type": "critic",   "platform": "rt"},
            "audience-reviews-imdb": {"type": "audience", "platform": "imdb"},
            # "critic-reviews-metacritic": {"type": "critic",   "platform": "metacritic"}
        }
        self.stats = {"scraped_files": 0, "dropped_no_finance": 0, "final_movies": 0}

    def clean_val(self, v):
        if pd.isna(v) or v == "N/A": return np.nan
        s = re.sub(r'[^\d.]', '', str(v))
        return float(s) if s else np.nan

    def slugify(self, filename):
        name = os.path.splitext(filename)[0].lower()
        for term in ["reviews", "imdb", "rt", "roger", "ebert", "metadata", "audience", "critic"]:
            name = name.replace(term, "")
        return name.strip("_")

    def load_master_metadata(self):
        meta_list = []
        for folder in self.folders.keys():
            path = os.path.join(self.base_path, folder)
            if not os.path.exists(path): continue
            meta_files = [f for f in os.listdir(path) if "metadata" in f.lower() and f.endswith(".csv")]
            for mf in meta_files:
                temp_df = pd.read_csv(os.path.join(path, mf))
                temp_df.columns = [c.lower().strip() for c in temp_df.columns]
                # Try to find a 'movie_key' or 'title' column
                key_col = 'movie_key' if 'movie_key' in temp_df.columns else temp_df.columns[0]
                temp_df['movie_key_clean'] = temp_df[key_col].apply(self.slugify)
                meta_list.append(temp_df[['movie_key_clean', 'budget', 'gross_us', 'year']])
        
        master = pd.concat(meta_list).drop_duplicates(subset=['movie_key_clean'])
        master['budget'] = master['budget'].apply(self.clean_val)
        master['gross'] = master['gross_us'].apply(self.clean_val)
        return master.dropna(subset=['budget', 'gross']).set_index('movie_key_clean')

    def run_full_extraction(self):
        master_meta = self.load_master_metadata()
        all_data = []

        for folder, info in self.folders.items():
            path = os.path.join(self.base_path, folder)
            if not os.path.exists(path): continue
            files = [f for f in os.listdir(path) if f.endswith(".csv") and "metadata" not in f.lower()]
            
            for f in files:
                self.stats["scraped_files"] += 1
                key = self.slugify(f)
                
                if key in master_meta.index:
                    m = master_meta.loc[key]
                    rdf = pd.read_csv(os.path.join(path, f))
                    rdf.columns = [c.lower().strip() for c in rdf.columns]
                    
                    # --- SMART SCORE DETECTION ---
                    # Look for columns named 'score', 'rating', 'user_rating', or 'stars'
                    score_col = None
                    for col in ['user_rating', 'score', 'rating', 'stars', 'user rating']:
                        if col in rdf.columns:
                            score_col = col
                            break
                    
                    if score_col:
                        rdf['user_rating'] = pd.to_numeric(rdf[score_col], errors='coerce')
                    else:
                        rdf['user_rating'] = np.nan # Create it if it doesn't exist
                    
                    rdf['movie_title'] = key
                    rdf['budget'] = m['budget']
                    rdf['gross'] = m['gross']
                    rdf['year'] = m['year']
                    rdf['is_critic'] = 1 if info['type'] == "critic" else 0
                    all_data.append(rdf)
                else:
                    self.stats["dropped_no_finance"] += 1
        
        full_df = pd.concat(all_data, ignore_index=True)
        full_df['label'] = (full_df['gross'] / full_df['budget'] >= 3.0).astype(int)
        self.stats["final_movies"] = full_df['movie_title'].nunique()
        return full_df

    def generate_trials(self, df):
        # We need to keep rows even if user_rating is NaN to maximize text data for Trial B
        clean_df = df.dropna(subset=['review_text']).copy()
        trial_a = clean_df.copy()
        
        trial_b = clean_df.groupby('movie_title').agg({
            'review_text': lambda x: " [SEP] ".join(str(i) for i in x.fillna("")[:10]),
            'budget': 'first',
            'year': 'first',
            'label': 'first',
            'user_rating': 'mean',
            'is_critic': 'mean'
        }).reset_index()
        
        return trial_a, trial_b

# RE-RUN EXTRACTION
manager = MovieWorkflowManager()
full_dataset = manager.run_full_extraction()
trial_a, trial_b = manager.generate_trials(full_dataset)

### the architecture (the classes)

In [7]:
class MovieDataset(Dataset):
    def __init__(self, texts, numeric, labels, tok):
        # We ensure numeric and labels are tensors immediately
        self.texts = texts
        self.numeric = torch.tensor(numeric, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tok = tok
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], max_length=256, truncation=True, padding="max_length", return_tensors="pt")
        return {
            "ids": e["input_ids"].squeeze(0), 
            "mask": e["attention_mask"].squeeze(0), 
            "num": self.numeric[i], 
            "lab": self.labels[i]
        }

class MovieClassifier(nn.Module):
    def __init__(self, n_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        # Your Freezing Logic: Only fine-tune the final layer
        for n, p in self.bert.named_parameters():
            if "encoder.layer.11" not in n: p.requires_grad = False
        self.num_bnch = nn.Sequential(nn.Linear(n_dim, 64), nn.ReLU())
        self.fusion = nn.Sequential(nn.Linear(768+64, 256), nn.ReLU(), nn.Linear(256, 1))
        
    def forward(self, ids, mask, num):
        # Using [CLS] token from last_hidden_state per your preference
        t_f = self.bert(ids, mask).last_hidden_state[:, 0, :]
        n_f = self.num_bnch(num)
        return self.fusion(torch.cat([t_f, n_f], dim=1)).squeeze(1)

### evaluate trial A and B 

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score

def compare_trials_initially(t_a, t_b):
    results = {}
    
    # UPDATE: Changed 'score' to 'user_rating' to match your data
    feats = ['user_rating', 'year'] 
    
    for name, df in [("Trial A (Individual)", t_a), ("Trial B (Collective)", t_b)]:
        # 1. Clean data for this trial
        # We also check if the columns actually exist before dropping
        existing_feats = [f for f in feats if f in df.columns]
        temp_df = df.dropna(subset=existing_feats + ['label'])
        
        # SAFETY CHECK: Print how many rows are left
        print(f"Checking {name}: {len(temp_df)} rows found with valid features.")
        
        if len(temp_df) == 0:
            print(f"⚠️ Skipping {name} because it has 0 rows. Check column names!")
            continue

        X = temp_df[existing_feats]
        y = temp_df['label']
        
        # 2. Split (Ensuring the same movie isn't in both Train and Test)
        gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
        
        try:
            train_idx, test_idx = next(gss.split(X, y, groups=temp_df['movie_title']))
            
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            # 3. Train a quick Random Forest (Method 1 Baseline)
            rf = RandomForestClassifier(n_estimators=100, random_state=42)
            rf.fit(X_train, y_train)
            
            # 4. Score it
            preds = rf.predict(X_test)
            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds, average='weighted')
            
            results[name] = {"Accuracy": acc, "F1-Score": f1}
            print(f"--- {name} Results ---")
            print(f"Accuracy: {acc:.3f} | F1-Score: {f1:.3f}\n")
        except Exception as e:
            print(f"Error processing {name}: {e}")

    # Determine the winner
    if not results:
        print("❌ ERROR: Both trials are empty. Check your data loading step.")
        return "None"
        
    winner = max(results, key=lambda x: results[x]['F1-Score'])
    print(f"🏆 THE BETTER DATA STRUCTURE IS: {winner}")
    return winner

# RUN THE COMPARISON
winning_trial_name = compare_trials_initially(trial_a, trial_b)

Checking Trial A (Individual): 0 rows found with valid features.
⚠️ Skipping Trial A (Individual) because it has 0 rows. Check column names!
Checking Trial B (Collective): 0 rows found with valid features.
⚠️ Skipping Trial B (Collective) because it has 0 rows. Check column names!
❌ ERROR: Both trials are empty. Check your data loading step.


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW # Corrected import location
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# ══════════════════════════════════════════════════════════════════════════════
# DATA PREPARATION FOR WINNING TRIAL
# ══════════════════════════════════════════════════════════════════════════════

# Select the winning data based on your previous cell
# winner_df is determined by your "Initial Score" logic
df_final = trial_b if "Trial B" in winning_trial_name else trial_a

# Split data (Ensuring no movie-leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df_final, groups=df_final['movie_title']))
train_df, test_df = df_final.iloc[train_idx].copy(), df_final.iloc[test_idx].copy()

# Features for Method 1
numeric_feats = ['user_rating', 'year', 'budget']

# ══════════════════════════════════════════════════════════════════════════════
# METHOD 1: RANDOM FOREST (Numeric Baseline Only)
# ══════════════════════════════════════════════════════════════════════════════
print("--- RUNNING METHOD 1: NUMERIC BASELINE ---")

scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_df[numeric_feats].fillna(0))
X_test_num = scaler.transform(test_df[numeric_feats].fillna(0))

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_num, train_df['label'])

m1_preds = rf_model.predict(X_test_num)
m1_f1 = f1_score(test_df['label'], m1_preds, average='weighted')

print(f"Method 1 (Numeric Only) F1-Score: {m1_f1:.3f}\n")

# ══════════════════════════════════════════════════════════════════════════════
# METHOD 2: BERT ONLY (Qualitative Baseline Only)
# ══════════════════════════════════════════════════════════════════════════════
print("--- RUNNING METHOD 2: BERT QUALITATIVE BASELINE ---")

class SimpleTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        encoding = self.tokenizer(self.texts[i], truncation=True, padding='max_length', max_length=256, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[i], dtype=torch.long)
        }

# Setup BERT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
m2_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2).to(device)

# Prepare DataLoaders
train_ds = SimpleTextDataset(train_df['review_text'].tolist(), train_df['label'].tolist(), tokenizer)
test_ds = SimpleTextDataset(test_df['review_text'].tolist(), test_df['label'].tolist(), tokenizer)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8)

# Optimizer (Using the corrected AdamW)
optimizer = AdamW(m2_model.parameters(), lr=2e-5)

# Training Loop (Method 2)
m2_model.train()
for epoch in range(2): 
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = m2_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} complete.")

# Evaluation (Method 2)
m2_model.eval()
m2_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = m2_model(input_ids, attention_mask=attention_mask)
        m2_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())

m2_f1 = f1_score(test_df['label'], m2_preds, average='weighted')
print(f"Method 2 (BERT Only) F1-Score: {m2_f1:.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# ECONOMIC CONFUSION MATRIX
# ══════════════════════════════════════════════════════════════════════════════

def plot_economic_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    # Manual check for binary cases
    if cm.shape == (1,1): # Handles edge cases where only one label is predicted
        print("Warning: Only one class predicted, Confusion Matrix plot skipped.")
        return
        
    labels = [
        ["True Negative\n(Avoided Disaster)", "False Positive\n(The Money Pit)"],
        ["False Negative\n(Sleeper Hit)", "True Positive\n(The Blockbuster)"]
    ]
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=["Predicted Flop", "Predicted Hit"],
                yticklabels=["Actual Flop", "Actual Hit"])
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j+0.5, i+0.7, labels[i][j], ha='center', va='center', fontsize=10, color='darkred')
            
    plt.title(title)
    plt.show()

plot_economic_matrix(test_df['label'], m2_preds, "Method 2: Qualitative Sentiment Confusion Matrix")

ValueError: With n_samples=1, test_size=0.3 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
# 2. METHOD 3: TRAINING THE HYBRID
# ══════════════════════════════════════════════════════════════════════════════
print("--- RUNNING METHOD 3: HYBRID (YOUR ARCHITECTURE) ---")

numeric_feats = ['user_rating', 'year', 'budget']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = BertTokenizer.from_pretrained("bert-base-uncased")

# Scaling numeric data
X_tr_num = scaler.fit_transform(train_df[numeric_feats].fillna(0))
X_te_num = scaler.transform(test_df[numeric_feats].fillna(0))

# Preparing DataLoaders using your MovieDataset
train_ldr = DataLoader(MovieDataset(train_df['review_text'].tolist(), X_tr_num, train_df['label'].tolist(), tok), batch_size=8, shuffle=True)
test_ldr = DataLoader(MovieDataset(test_df['review_text'].tolist(), X_te_num, test_df['label'].tolist(), tok), batch_size=8)

# Initializing your MovieClassifier
model_hybrid = MovieClassifier(len(numeric_feats)).to(device)
opt = AdamW(filter(lambda p: p.requires_grad, model_hybrid.parameters()), lr=3e-5)
crit = nn.BCEWithLogitsLoss()

# Training loop
model_hybrid.train()
for ep in range(3):
    total_loss = 0
    for b in train_ldr:
        opt.zero_grad()
        out = model_hybrid(b["ids"].to(device), b["mask"].to(device), b["num"].to(device))
        loss = crit(out, b["lab"].to(device))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {ep+1} | Loss: {total_loss/len(train_ldr):.4f}")

# Evaluation
model_hybrid.eval()
m3_probs = []
with torch.no_grad():
    for b in test_ldr:
        logits = model_hybrid(b["ids"].to(device), b["mask"].to(device), b["num"].to(device))
        m3_probs.extend(torch.sigmoid(logits).cpu().numpy())

m3_preds = (np.array(m3_probs) >= 0.5).astype(int)
m3_f1 = f1_score(test_df['label'], m3_preds, average='weighted')
print(f"Method 3 (Hybrid) F1-Score: {m3_f1:.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. METHOD 4: THE ENSEMBLE (Final Step)
# ══════════════════════════════════════════════════════════════════════════════
print("\n--- RUNNING METHOD 4: THE ENSEMBLE ---")

# Pull probabilities from Method 1 (Random Forest)
m1_probs = rf_model.predict_proba(X_te_num)[:, 1]

# Average Method 1 and Method 3
m4_probs = (np.array(m1_probs) + np.array(m3_probs)) / 2
m4_preds = (m4_probs >= 0.5).astype(int)
m4_f1 = f1_score(test_df['label'], m4_preds, average='weighted')

print(f"Method 4 (Ensemble) F1-Score: {m4_f1:.3f}")

# Final "Discovery" Plot for Section 1.3
plt.figure(figsize=(10, 5))
methods = ['M1: Numeric', 'M2: BERT-Only', 'M3: Hybrid', 'M4: Ensemble']
scores = [m1_f1, m2_f1, m3_f1, m4_f1]
sns.barplot(x=methods, y=scores, palette="rocket")
plt.axhline(y=test_df['label'].value_counts(normalize=True).max(), color='red', linestyle='--', label='Majority Class Baseline')
plt.title("Final Comparison: Which Method Best Predicts Financial Success?")
plt.ylabel("Weighted F1-Score")
plt.ylim(0, 1.0)
plt.legend()
plt.show()